# DefectosCafeVerde — train-fitted routed validation screen
Jalankan hanya setelah audit train RGB-native v2 memberikan keputusan AUTHORIZE. Router difit dari train records sebelum validation dibaca. Tidak ada detector training dan test tidak diekstrak.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import importlib, json, os, shutil, subprocess, sys, tarfile
WORK=Path('/content'); REPO=WORK/'coffee-bean-detection'
BRANCH='codex/defectoscafeverde-error-complementarity'
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True,cwd=WORK)
import torch
print('TORCH PRE-INSTALL:',torch.__version__,'| CUDA BUILD:',torch.version.cuda,'| CUDA AVAILABLE:',torch.cuda.is_available())
if not torch.cuda.is_available():
    subprocess.run(['nvidia-smi'],check=False)
    raise RuntimeError('PyTorch GPU tidak tersedia sebelum instalasi. Restart runtime dan pastikan T4 terhubung.')
torch_version_before=torch.__version__
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','ultralytics==8.4.96','ultralytics-thop'],check=True,cwd=WORK)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True,cwd=WORK)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
if torch.__version__!=torch_version_before or not torch.cuda.is_available(): raise RuntimeError('Instalasi mengubah PyTorch CUDA; restart runtime.')
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
from coffee_detector.experiments.prepare_defectoscafeverde_routability_colab import prepare_inputs
PROJECT,ARCHIVE,RESULTS,CHECKPOINTS,SOURCES=prepare_inputs(WORK,'/content/drive')
TRAIN_OUT=PROJECT/'experiments/defectoscafeverde-train-routability-v2-rgb-native'
TRAIN_SUMMARY=TRAIN_OUT/'defectoscafeverde_train_routability.json'
TRAIN_RECORDS=TRAIN_OUT/'defectoscafeverde_train_routability_targets.jsonl'
if not TRAIN_SUMMARY.is_file() or not TRAIN_RECORDS.is_file(): raise FileNotFoundError('Audit train RGB-native v2 belum lengkap; jalankan notebook Train Routability terbaru dahulu.')
train_summary=json.loads(TRAIN_SUMMARY.read_text())
if train_summary.get('decision')!='AUTHORIZE_SEPARATE_ROUTED_ARCHITECTURE_SCREEN': raise RuntimeError(f'Audit train tidak mengotorisasi validation: {train_summary.get("decision")}')
OUT=PROJECT/'experiments/defectoscafeverde-routed-validation-v1'; OUT.mkdir(parents=True,exist_ok=True)
print('TRAIN DECISION:',train_summary['decision'])
print('SUBSET:',train_summary['minimal_subset']['subset'])
print('OUTPUT:',OUT)

In [ ]:
import yaml
DATA=WORK/'defectoscafeverde-routed-validation'
if DATA.exists(): shutil.rmtree(DATA)
DATA.mkdir(parents=True)
prefix='grouped-physical-v1/'; allowed={'data.yaml','grouped_audit.json','grouped_summary.json'}
with tarfile.open(ARCHIVE,'r') as bundle:
    for member in bundle.getmembers():
        name=member.name.replace('\\','/')
        if not name.startswith(prefix): continue
        relative=name[len(prefix):].strip('/')
        if not relative: continue
        first=relative.split('/',1)[0]
        if not (first=='val' or relative in allowed): continue
        target=(DATA/relative).resolve()
        if DATA.resolve() not in target.parents and target!=DATA.resolve(): raise RuntimeError('Unsafe archive path')
        if member.isdir(): target.mkdir(parents=True,exist_ok=True); continue
        target.parent.mkdir(parents=True,exist_ok=True); source=bundle.extractfile(member)
        if source is None: raise RuntimeError(f'Gagal membaca {member.name}')
        with source,target.open('wb') as output: shutil.copyfileobj(source,output)
if (DATA/'train').exists() or (DATA/'test').exists(): raise RuntimeError('TRAIN/TEST TEREXPOSE - STOP')
payload=yaml.safe_load((DATA/'data.yaml').read_text()); payload.pop('train',None); payload.pop('test',None)
payload['path']=str(DATA); payload['val']='val/images'
(DATA/'data.yaml').write_text(yaml.safe_dump(payload,sort_keys=False),encoding='utf-8')
print('VAL:',len(list((DATA/'val/images').glob('*'))),'TRAIN:',(DATA/'train').exists(),'TEST:',(DATA/'test').exists())

In [ ]:
from coffee_detector.analysis.defectoscafeverde_routed_validation import run_routed_validation_screen
SUMMARY=OUT/'defectoscafeverde_routed_validation.json'
OBSERVATIONS=OUT/'defectoscafeverde_routed_validation_observations.jsonl'
ROUTER=OUT/'defectoscafeverde_ridge_router.json'
result=run_routed_validation_screen(DATA,RESULTS,CHECKPOINTS,TRAIN_SUMMARY,TRAIN_RECORDS,SUMMARY,observations_output=OBSERVATIONS,router_output=ROUTER,device='0')
print('SUMMARY:',SUMMARY)
print('ROUTER:',ROUTER)

In [ ]:
import pandas as pd
rows=[]
for model,metrics in result['results'].items():
    rows.append({'model':model,**{key:metrics[key] for key in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')},'worst_class':metrics['worst_class']})
display(pd.DataFrame(rows).set_index('model').style.format({key:'{:.2%}' for key in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')}))
print('SUBSET:',result['subset'])
print('SELECTION COUNTS:',result['selection_counts'])
print('CALIBRATION:',result['calibration_vs_historical_d0'])
print('DELTAS:',result['routed_minus_runtime_d0'])
print('CRITERIA:',result['criteria'])
print('DECISION:',result['decision'])
print('NEXT:',result['next'])
print('DETECTOR TRAINING:',result['detector_training_executed'],'| ROUTER TRAIN-ONLY:',result['router_fitted_from_train_only'],'| TEST:',result['test_images_accessed'])
print('Kirim tabel, SUBSET, SELECTION COUNTS, CALIBRATION, DELTAS, CRITERIA, DECISION, dan NEXT. Jangan membuka test.')